# Taller 10: Autocorrelación y Series Temporales

En este taller exploraremos el fenómeno de la autocorrelación en series temporales y cómo abordarla desde el marco de los modelos lineales. Estudiaremos métodos para detectar, cuantificar y corregir la autocorrelación, así como técnicas específicas para el análisis de datos con estructura temporal.

## Objetivos de Aprendizaje
- Comprender el concepto de autocorrelación y sus implicaciones para los modelos lineales
- Implementar pruebas diagnósticas para detectar autocorrelación
- Aplicar métodos de corrección como Cochrane-Orcutt y Prais-Winsten
- Analizar series temporales mediante modelos de regresión
- Interpretar correctamente los resultados en presencia de autocorrelación


In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.stattools import acf, pacf, adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox, acorr_breusch_godfrey
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.regression.linear_model import OLS, GLS
from statsmodels.stats.stattools import durbin_watson
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import warnings

# Configuración para visualizaciones
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('talk')
warnings.filterwarnings('ignore')
%matplotlib inline


# Fundamentos Teóricos de Autocorrelación y Series Temporales

## 1. Autocorrelación en Modelos Lineales

La autocorrelación (también llamada correlación serial) ocurre cuando los errores de un modelo de regresión están correlacionados a lo largo del tiempo o de observaciones secuenciales. Formalmente, si denotamos el error en el tiempo $t$ como $\varepsilon_t$, la autocorrelación implica que:

$$Cov(\varepsilon_t, \varepsilon_{t-k}) \neq 0 \quad \text{para algún } k \neq 0$$

### 1.1 Modelo de Autocorrelación de Primer Orden (AR(1))

El modelo más común de autocorrelación es el proceso autorregresivo de primer orden, AR(1):

$$\varepsilon_t = \rho \varepsilon_{t-1} + u_t$$

donde:
- $\rho$ es el coeficiente de autocorrelación (entre -1 y 1)
- $u_t$ es un término de error que sigue ruido blanco (independiente e idénticamente distribuido)

### 1.2 Consecuencias de Ignorar la Autocorrelación

Cuando existe autocorrelación y se ignora:
- Los estimadores OLS siguen siendo insesgados y consistentes, pero ya no son eficientes
- Los errores estándar estimados son incorrectos (generalmente subestimados)
- Las pruebas de hipótesis (t, F) y los intervalos de confianza son inválidos
- La predicción es subóptima

## 2. Detección de Autocorrelación

Existen varios métodos para detectar la presencia de autocorrelación:

### 2.1 Prueba de Durbin-Watson

La prueba de Durbin-Watson es una de las más utilizadas para detectar autocorrelación de primer orden:

$$DW = \frac{\sum_{t=2}^{T} (e_t - e_{t-1})^2}{\sum_{t=1}^{T} e_t^2}$$

donde $e_t$ son los residuos del modelo OLS.

- Si $DW \approx 2$, no hay evidencia de autocorrelación
- Si $DW < 2$, existe autocorrelación positiva
- Si $DW > 2$, existe autocorrelación negativa

### 2.2 Función de Autocorrelación (ACF) y Función de Autocorrelación Parcial (PACF)

- **ACF**: Mide la correlación entre una serie y sus rezagos para diferentes órdenes
- **PACF**: Mide la correlación entre una serie y sus rezagos, controlando por los efectos de rezagos intermedios

### 2.3 Prueba de Breusch-Godfrey

La prueba de Breusch-Godfrey es una prueba más general que puede detectar autocorrelación de orden superior:

1. Se ajusta un modelo OLS y se obtienen los residuos $e_t$
2. Se regresa $e_t$ sobre las variables originales y los residuos rezagados $e_{t-1}, e_{t-2}, \ldots, e_{t-p}$
3. Se prueba la significancia conjunta de los coeficientes de los residuos rezagados

## 3. Corrección de Autocorrelación

Hay varios métodos para corregir la autocorrelación:

### 3.1 Método de Cochrane-Orcutt

1. Estimar el modelo OLS y obtener los residuos
2. Estimar $\rho$ regresando $e_t$ sobre $e_{t-1}$
3. Transformar las variables: $y_t^* = y_t - \hat{\rho}y_{t-1}$ y $x_{it}^* = x_{it} - \hat{\rho}x_{i,t-1}$
4. Estimar el modelo con las variables transformadas
5. Repetir los pasos 2-4 hasta que $\hat{\rho}$ converja

### 3.2 Método de Prais-Winsten

Similar a Cochrane-Orcutt, pero conserva la primera observación aplicando una transformación especial, lo que mejora la eficiencia cuando la muestra es pequeña.

### 3.3 Modelos ARIMA

Para series temporales con patrones más complejos, los modelos ARIMA (AutoRegressive Integrated Moving Average) combinan:
- Componentes autorregresivos (AR)
- Diferenciación para lograr estacionariedad (I)
- Componentes de medias móviles (MA)

## 4. Series Temporales y Modelos Lineales

Al aplicar modelos lineales a series temporales, debemos considerar:

### 4.1 Estacionariedad

Una serie temporal es estacionaria si sus propiedades estadísticas (media, varianza, autocorrelación) no cambian con el tiempo. La mayoría de las técnicas de series temporales asumen estacionariedad.

### 4.2 Tendencias y Estacionalidad

- **Tendencia**: Componente a largo plazo que representa el aumento o disminución en la serie
- **Estacionalidad**: Patrones cíclicos que se repiten en intervalos regulares

### 4.3 Regresión con Variables Rezagadas

Se pueden incluir valores rezagados de la variable dependiente o independientes como predictores:

$$y_t = \beta_0 + \beta_1 x_{1t} + \beta_2 y_{t-1} + \varepsilon_t$$

Esto captura la dependencia temporal pero puede introducir problemas de endogeneidad.


# Implementación de Pruebas de Autocorrelación

A continuación implementaremos funciones para detectar y visualizar la autocorrelación en los residuos de un modelo de regresión.


In [ ]:

def diagnostico_autocorrelacion(modelo, max_rezago=20):
    """
    Realiza diagnósticos de autocorrelación en los residuos de un modelo.
    
    Parámetros:
    -----------
    modelo : objeto de modelo ajustado de statsmodels
        Modelo de regresión ajustado
    max_rezago : int, opcional (default=20)
        Número máximo de rezagos para análisis
        
    Retorna:
    --------
    resultados : dict
        Diccionario con resultados de las pruebas
    """
    # Obtener residuos
    residuos = modelo.resid
    
    # Prueba de Durbin-Watson
    dw = durbin_watson(residuos)
    
    # Prueba de Breusch-Godfrey para autocorrelación de orden superior
    bg_test = acorr_breusch_godfrey(modelo, nlags=min(max_rezago, len(residuos) // 5))
    
    # Prueba de Ljung-Box
    lb_test = acorr_ljungbox(residuos, lags=min(max_rezago, len(residuos) // 5))
    
    # Función de autocorrelación y autocorrelación parcial
    acf_values = acf(residuos, nlags=max_rezago, fft=True)
    pacf_values = pacf(residuos, nlags=max_rezago)
    
    # Recopilar resultados
    resultados = {
        'durbin_watson': dw,
        'breusch_godfrey': {
            'lm_stat': bg_test[0],
            'p_valor': bg_test[1],
            'conclusion': 'Autocorrelación' if bg_test[1] < 0.05 else 'No autocorrelación'
        },
        'ljung_box': {
            'lb_stat': list(lb_test['lb_stat']),
            'p_valor': list(lb_test['lb_pvalue']),
            'conclusion': list(['Autocorrelación' if p < 0.05 else 'No autocorrelación' for p in lb_test['lb_pvalue']])
        },
        'acf': acf_values,
        'pacf': pacf_values
    }
    
    return resultados

def visualizar_autocorrelacion(residuos, max_rezago=20):
    """
    Visualiza la autocorrelación en los residuos mediante gráficos ACF y PACF.
    
    Parámetros:
    -----------
    residuos : array o Series
        Residuos del modelo
    max_rezago : int, opcional (default=20)
        Número máximo de rezagos para visualizar
        
    Retorna:
    --------
    fig : objeto Figure de matplotlib
        Figura con los gráficos ACF y PACF
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Gráfico de residuos vs tiempo
    axes[0, 0].plot(residuos)
    axes[0, 0].set_title('Residuos vs Tiempo')
    axes[0, 0].set_xlabel('Tiempo')
    axes[0, 0].set_ylabel('Residuos')
    axes[0, 0].axhline(y=0, color='r', linestyle='-')
    
    # Gráfico de residuos rezagados (t vs t-1)
    if len(residuos) > 1:
        axes[0, 1].scatter(residuos[:-1], residuos[1:], alpha=0.5)
        axes[0, 1].set_title('Residuos(t) vs Residuos(t-1)')
        axes[0, 1].set_xlabel('Residuos(t-1)')
        axes[0, 1].set_ylabel('Residuos(t)')
        
        # Añadir línea de regresión
        z = np.polyfit(residuos[:-1], residuos[1:], 1)
        p = np.poly1d(z)
        axes[0, 1].plot(sorted(residuos[:-1]), p(sorted(residuos[:-1])), "r--", alpha=0.8)
        
        # Añadir valor estimado de ρ
        rho = np.corrcoef(residuos[:-1], residuos[1:])[0, 1]
        axes[0, 1].annotate(f'ρ = {rho:.3f}', xy=(0.05, 0.95), xycoords='axes fraction',
                            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.8))
    
    # Gráfico ACF
    plot_acf(residuos, ax=axes[1, 0], lags=max_rezago, alpha=0.05)
    axes[1, 0].set_title('Función de Autocorrelación (ACF)')
    
    # Gráfico PACF
    plot_pacf(residuos, ax=axes[1, 1], lags=max_rezago, alpha=0.05)
    axes[1, 1].set_title('Función de Autocorrelación Parcial (PACF)')
    
    plt.tight_layout()
    return fig

def reportar_diagnostico_autocorrelacion(resultados):
    """
    Genera un reporte con los resultados del diagnóstico de autocorrelación.
    
    Parámetros:
    -----------
    resultados : dict
        Diccionario con resultados obtenidos de diagnostico_autocorrelacion()
        
    Retorna:
    --------
    None
        Imprime un reporte con los resultados
    """
    print("=" * 50)
    print("DIAGNÓSTICO DE AUTOCORRELACIÓN")
    print("=" * 50)
    
    # Estadístico Durbin-Watson
    dw = resultados['durbin_watson']
    print(f"Estadístico Durbin-Watson: {dw:.4f}")
    
    if dw < 1.5:
        conclusion = "Evidencia de autocorrelación positiva"
    elif dw > 2.5:
        conclusion = "Evidencia de autocorrelación negativa"
    else:
        conclusion = "No hay evidencia fuerte de autocorrelación de primer orden"
    
    print(f"Interpretación: {conclusion}")
    print("-" * 50)
    
    # Prueba de Breusch-Godfrey
    bg = resultados['breusch_godfrey']
    print(f"Prueba de Breusch-Godfrey para autocorrelación de orden superior:")
    print(f"  Estadístico LM: {bg['lm_stat']:.4f}")
    print(f"  Valor p: {bg['p_valor']:.4f}")
    print(f"  Conclusión: {bg['conclusion']}")
    print("-" * 50)
    
    # Prueba de Ljung-Box para los primeros 5 rezagos
    lb = resultados['ljung_box']
    print("Prueba de Ljung-Box (primeros 5 rezagos):")
    for i in range(min(5, len(lb['lb_stat']))):
        print(f"  Rezago {i+1}: Estadístico={lb['lb_stat'][i]:.4f}, p-valor={lb['p_valor'][i]:.4f}, {lb['conclusion'][i]}")
    
    print("\nNota: La prueba de Durbin-Watson es específica para autocorrelación de primer orden.")
    print("Para detectar autocorrelación de orden superior, utilizar Breusch-Godfrey o Ljung-Box.")


# Implementación de Métodos de Corrección de Autocorrelación

A continuación, implementaremos desde cero los principales métodos para corregir la autocorrelación en modelos de regresión.


In [ ]:

def estimar_rho(residuos):
    """
    Estima el coeficiente de autocorrelación de primer orden (ρ).
    
    Parámetros:
    -----------
    residuos : array o Series
        Residuos del modelo
        
    Retorna:
    --------
    rho : float
        Coeficiente de autocorrelación estimado
    """
    # Asegurar que residuos sea un array de numpy
    if isinstance(residuos, pd.Series):
        residuos = residuos.values
    
    # Calcular ρ como la correlación entre residuos(t) y residuos(t-1)
    rho = np.corrcoef(residuos[:-1], residuos[1:])[0, 1]
    
    return rho

def cochrane_orcutt(y, X, max_iter=20, tol=1e-5):
    """
    Implementa el método de Cochrane-Orcutt para corregir autocorrelación.
    
    Parámetros:
    -----------
    y : array o Series
        Variable dependiente
    X : array o DataFrame
        Variables independientes
    max_iter : int, opcional (default=20)
        Número máximo de iteraciones
    tol : float, opcional (default=1e-5)
        Tolerancia para convergencia
        
    Retorna:
    --------
    resultados : dict
        Diccionario con el modelo final y otros resultados
    """
    # Asegurar formatos adecuados
    if isinstance(y, pd.Series):
        y = y.values
    
    if isinstance(X, pd.DataFrame):
        X_nombres = X.columns
        X = X.values
    else:
        X_nombres = [f'X{i+1}' for i in range(X.shape[1])]
    
    # Añadir constante si no está presente
    if not np.all(X[:, 0] == 1):
        X = sm.add_constant(X)
        X_nombres = ['const'] + list(X_nombres)
    
    # Modelo OLS inicial
    modelo_ols = sm.OLS(y, X).fit()
    rho_old = 0
    rho = estimar_rho(modelo_ols.resid)
    
    iter_count = 0
    modelos = [modelo_ols]
    rhos = [rho]
    
    # Proceso iterativo
    while abs(rho - rho_old) > tol and iter_count < max_iter:
        # Transformar variables (excepto la primera observación)
        y_transformed = y[1:] - rho * y[:-1]
        X_transformed = X[1:] - rho * X[:-1]
        
        # Ajustar modelo con variables transformadas
        modelo = sm.OLS(y_transformed, X_transformed).fit()
        
        # Actualizar rho
        rho_old = rho
        rho = estimar_rho(modelo.resid)
        
        modelos.append(modelo)
        rhos.append(rho)
        iter_count += 1
    
    # Recopilar resultados
    resultados = {
        'modelo_original': modelo_ols,
        'modelo_final': modelos[-1],
        'rho_final': rhos[-1],
        'n_iteraciones': iter_count,
        'convergencia': abs(rho - rho_old) <= tol,
        'rhos_historia': rhos,
        'X_nombres': X_nombres
    }
    
    return resultados

def prais_winsten(y, X, max_iter=20, tol=1e-5):
    """
    Implementa el método de Prais-Winsten para corregir autocorrelación.
    
    Parámetros:
    -----------
    y : array o Series
        Variable dependiente
    X : array o DataFrame
        Variables independientes
    max_iter : int, opcional (default=20)
        Número máximo de iteraciones
    tol : float, opcional (default=1e-5)
        Tolerancia para convergencia
        
    Retorna:
    --------
    resultados : dict
        Diccionario con el modelo final y otros resultados
    """
    # Asegurar formatos adecuados
    if isinstance(y, pd.Series):
        y = y.values
    
    if isinstance(X, pd.DataFrame):
        X_nombres = X.columns
        X = X.values
    else:
        X_nombres = [f'X{i+1}' for i in range(X.shape[1])]
    
    # Añadir constante si no está presente
    if not np.all(X[:, 0] == 1):
        X = sm.add_constant(X)
        X_nombres = ['const'] + list(X_nombres)
    
    # Modelo OLS inicial
    modelo_ols = sm.OLS(y, X).fit()
    rho_old = 0
    rho = estimar_rho(modelo_ols.resid)
    
    iter_count = 0
    modelos = [modelo_ols]
    rhos = [rho]
    
    # Proceso iterativo
    while abs(rho - rho_old) > tol and iter_count < max_iter:
        # Transformar la primera observación
        y_transformed_first = y[0] * np.sqrt(1 - rho**2)
        X_transformed_first = X[0] * np.sqrt(1 - rho**2)
        
        # Transformar el resto de observaciones
        y_transformed_rest = y[1:] - rho * y[:-1]
        X_transformed_rest = X[1:] - rho * X[:-1]
        
        # Combinar
        y_transformed = np.concatenate(([y_transformed_first], y_transformed_rest))
        X_transformed = np.vstack((X_transformed_first, X_transformed_rest))
        
        # Ajustar modelo con variables transformadas
        modelo = sm.OLS(y_transformed, X_transformed).fit()
        
        # Actualizar rho
        rho_old = rho
        # Para Prais-Winsten, necesitamos calcular los residuos originales para estimar rho
        y_pred = modelo.predict(X)
        residuos = y - y_pred
        rho = estimar_rho(residuos)
        
        modelos.append(modelo)
        rhos.append(rho)
        iter_count += 1
    
    # Recopilar resultados
    resultados = {
        'modelo_original': modelo_ols,
        'modelo_final': modelos[-1],
        'rho_final': rhos[-1],
        'n_iteraciones': iter_count,
        'convergencia': abs(rho - rho_old) <= tol,
        'rhos_historia': rhos,
        'X_nombres': X_nombres
    }
    
    return resultados


def comparar_modelos_autocorrelacion(y, X):
    """
    Compara diferentes modelos para corregir autocorrelación.

    Parámetros:
    -----------
    y : array o Series
        Variable dependiente
    X : array o DataFrame
        Variables independientes

    Retorna:
    --------
    comparacion : DataFrame
        DataFrame con comparación de modelos
    """
    import pandas as pd
    import numpy as np
    import statsmodels.api as sm
    from statsmodels.stats.stattools import durbin_watson

    # Asegurar que X sea DataFrame y agregar constante
    X_df = pd.DataFrame(X) if isinstance(X, np.ndarray) else X.copy()
    X_sm = sm.add_constant(X_df)

    # Modelo OLS
    ols = sm.OLS(y, X_sm).fit()
    rho_ols = estimar_rho(ols.resid)
    dw_ols = durbin_watson(ols.resid)

    # ==========================
    # Cochrane-Orcutt
    # ==========================
    co_results = cochrane_orcutt(y, X_df)
    rho_co = co_results['rho_final']
    modelo_co = co_results['modelo_final']

    # Transformar X para predicción
    X_co = pd.DataFrame(
        X_sm[1:] - rho_co * X_sm[:-1],
        index=np.arange(len(y) - 1),
        columns=X_sm.columns
    )
    # Predecir
    preds_co = modelo_co.predict(X_co)
    y_pred_co = np.full_like(y, np.nan)
    y_pred_co[1:] = preds_co

    # Métricas Cochrane-Orcutt
    residuos_co = y[1:] - y_pred_co[1:]
    ssr_co = np.sum(residuos_co ** 2)
    sst_co = np.sum((y[1:] - np.mean(y[1:])) ** 2)
    r2_co = 1 - ssr_co / sst_co
    dw_co = durbin_watson(residuos_co)

    # ==========================
    # Prais-Winsten
    # ==========================
    pw_results = prais_winsten(y, X_df)
    rho_pw = pw_results['rho_final']
    modelo_pw = pw_results['modelo_final']

    # Transformar X para Prais-Winsten
    X_pw_0 = X_sm.iloc[0:1] * np.sqrt(1 - rho_pw**2)
    X_pw_rest = X_sm.iloc[1:] - rho_pw * X_sm.iloc[:-1].values
    X_pw = pd.concat([X_pw_0, pd.DataFrame(X_pw_rest, columns=X_sm.columns)], ignore_index=True)

    # Predecir
    y_pred_pw = modelo_pw.predict(X_pw)
    residuos_pw = y - y_pred_pw
    ssr_pw = np.sum(residuos_pw ** 2)
    sst_pw = np.sum((y - np.mean(y)) ** 2)
    r2_pw = 1 - ssr_pw / sst_pw
    dw_pw = durbin_watson(residuos_pw)

    # ==========================
    # GLS con estructura AR(1) (comparativo)
    # ==========================
    # Nota: aquí usamos la matriz de covarianza de errores como identidad (sin autocorrelación)
    gls = sm.GLS(y, X_sm).fit()
    dw_gls = durbin_watson(gls.resid)

    # ==========================
    # Crear DataFrame de comparación
    # ==========================
    metodos = ['OLS', 'Cochrane-Orcutt', 'Prais-Winsten', 'GLS-AR(1)']
    rhos = [rho_ols, rho_co, rho_pw, rho_ols]  # en GLS no se estimó rho real
    r2s = [ols.rsquared, r2_co, r2_pw, gls.rsquared]
    dws = [dw_ols, dw_co, dw_pw, dw_gls]
    iteraciones = [0, co_results['n_iteraciones'], pw_results['n_iteraciones'], 0]

    comparacion = pd.DataFrame({
        'Método': metodos,
        'Rho Estimado': rhos,
        'R²': r2s,
        'Durbin-Watson': dws,
        'Iteraciones': iteraciones
    })

    return comparacion


def reportar_resultados_correccion(resultados, metodo="Cochrane-Orcutt"):
    """
    Reporta los resultados de la corrección por autocorrelación.
    
    Parámetros:
    -----------
    resultados : dict
        Resultados de cochrane_orcutt() o prais_winsten()
    metodo : str, opcional (default="Cochrane-Orcutt")
        Nombre del método utilizado
        
    Retorna:
    --------
    None
        Imprime un reporte con los resultados
    """
    print("=" * 50)
    print(f"RESULTADOS DE CORRECCIÓN POR {metodo.upper()}")
    print("=" * 50)
    
    modelo_orig = resultados['modelo_original']
    modelo_final = resultados['modelo_final']
    rho_final = resultados['rho_final']
    n_iter = resultados['n_iteraciones']
    convergencia = resultados['convergencia']
    X_nombres = resultados['X_nombres']
    
    print(f"Rho final estimado: {rho_final:.4f}")
    print(f"Número de iteraciones: {n_iter}")
    print(f"Convergencia: {'Sí' if convergencia else 'No'}")
    print()
    
    print("Comparación de coeficientes:")
    print("-" * 50)
    print(f"{'Variable':<15} {'Original':<15} {'Corregido':<15} {'% Cambio':<15}")
    print("-" * 50)
    
    coef_orig = modelo_orig.params
    
    if hasattr(modelo_final, 'params'):
        coef_final = modelo_final.params
        
        for i, nombre in enumerate(X_nombres):
            coef_o = coef_orig[i]
            coef_f = coef_final[i]
            pct_cambio = (coef_f - coef_o) / coef_o * 100 if coef_o != 0 else np.inf
            
            print(f"{nombre:<15} {coef_o:<15.4f} {coef_f:<15.4f} {pct_cambio:<15.2f}")
    
    print("\nDiagnóstico de autocorrelación:")
    print(f"DW original: {durbin_watson(modelo_orig.resid):.4f}")
    
    if metodo == "Cochrane-Orcutt":
        # Para Cochrane-Orcutt necesitamos calcular residuos transformados
        print(f"DW corregido: {durbin_watson(modelo_final.resid):.4f}")
    else:
        # Para Prais-Winsten necesitamos reconstruir los residuos
        print(f"DW corregido: {durbin_watson(modelo_final.resid):.4f}")
    
    print("\nHistoria de estimaciones de rho:")
    for i, r in enumerate(resultados['rhos_historia']):
        print(f"Iteración {i}: rho = {r:.4f}")


# Generación de Datos Sintéticos con Autocorrelación

Para entender mejor los efectos de la autocorrelación y evaluar los métodos de corrección, generaremos datos sintéticos con diferentes patrones de autocorrelación.


In [ ]:

def generar_datos_ar1(n=200, beta_true=[2.5, 1.5, -0.8], rho=0.7, seed=42):
    """
    Genera datos con estructura autorregresiva de primer orden (AR(1)).
    
    Parámetros:
    -----------
    n : int, opcional (default=200)
        Número de observaciones
    beta_true : list, opcional
        Coeficientes verdaderos [intercepto, beta_1, beta_2]
    rho : float, opcional (default=0.7)
        Coeficiente de autocorrelación para errores
    seed : int, opcional (default=42)
        Semilla para reproducibilidad
        
    Retorna:
    --------
    X : DataFrame
        Variables predictoras
    y : Series
        Variable respuesta
    errores : Series
        Errores del modelo
    """
    np.random.seed(seed)
    
    # Generar predictores 
    X1 = np.random.normal(0, 1, n)
    X2 = np.random.normal(0, 1, n)
    
    # Convertir a DataFrame para mejor manejo
    X = pd.DataFrame({'X1': X1, 'X2': X2})
    
    # Generar errores AR(1)
    errores = np.zeros(n)
    u = np.random.normal(0, 1, n)  # Innovaciones (ruido blanco)
    
    # Generar proceso AR(1)
    for t in range(1, n):
        errores[t] = rho * errores[t-1] + u[t]
    
    # Generar variable respuesta
    intercepto = beta_true[0]
    y = intercepto + beta_true[1] * X['X1'] + beta_true[2] * X['X2'] + errores
    
    # Convertir a Series
    y = pd.Series(y, name='Y')
    errores = pd.Series(errores, name='error')
    
    return X, y, errores

def generar_datos_tendencia_estacionalidad(n=200, beta_true=[2.5, 1.5], rho=0.7, 
                                          tendencia=0.1, amplitud_estacional=2.0, 
                                          periodo=12, seed=42):
    """
    Genera datos de series temporales con tendencia, estacionalidad y autocorrelación.
    
    Parámetros:
    -----------
    n : int, opcional (default=200)
        Número de observaciones
    beta_true : list, opcional
        Coeficientes verdaderos [intercepto, beta_1]
    rho : float, opcional (default=0.7)
        Coeficiente de autocorrelación para errores
    tendencia : float, opcional (default=0.1)
        Pendiente de la tendencia
    amplitud_estacional : float, opcional (default=2.0)
        Amplitud del patrón estacional
    periodo : int, opcional (default=12)
        Período del patrón estacional (ej. 12 meses, 4 trimestres)
    seed : int, opcional (default=42)
        Semilla para reproducibilidad
        
    Retorna:
    --------
    df : DataFrame
        DataFrame con todas las variables (tiempo, X, y, componentes)
    """
    np.random.seed(seed)
    
    # Crear índice temporal
    tiempo = np.arange(n)
    
    # Generar componente de tendencia
    tendencia_comp = tendencia * tiempo
    
    # Generar componente estacional
    estacional_comp = amplitud_estacional * np.sin(2 * np.pi * tiempo / periodo)
    
    # Generar predictor con autocorrelación
    X1 = np.zeros(n)
    u_x = np.random.normal(0, 1, n)
    for t in range(1, n):
        X1[t] = 0.5 * X1[t-1] + u_x[t]
    
    # Generar errores AR(1)
    errores = np.zeros(n)
    u = np.random.normal(0, 1, n)  # Innovaciones (ruido blanco)
    for t in range(1, n):
        errores[t] = rho * errores[t-1] + u[t]
    
    # Generar variable respuesta
    intercepto = beta_true[0]
    y_sin_error = intercepto + beta_true[1] * X1 + tendencia_comp + estacional_comp
    y = y_sin_error + errores
    
    # Crear DataFrame
    df = pd.DataFrame({
        'tiempo': tiempo,
        'X1': X1,
        'Y': y,
        'tendencia': tendencia_comp,
        'estacional': estacional_comp,
        'error': errores,
        'Y_sin_error': y_sin_error
    })
    
    return df

def visualizar_datos_autocorrelacion(X, y, errores, rho_true):
    """
    Visualiza datos generados con autocorrelación.
    
    Parámetros:
    -----------
    X : DataFrame
        Variables predictoras
    y : Series
        Variable respuesta
    errores : Series
        Errores del modelo
    rho_true : float
        Coeficiente verdadero de autocorrelación
        
    Retorna:
    --------
    fig : objeto Figure de matplotlib
        Figura con las visualizaciones
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Serie temporal de errores
    axes[0, 0].plot(errores)
    axes[0, 0].set_title('Errores a lo largo del tiempo')
    axes[0, 0].set_xlabel('Tiempo')
    axes[0, 0].set_ylabel('Error')
    
    # Diagrama de dispersión de errores(t) vs errores(t-1)
    axes[0, 1].scatter(errores[:-1], errores[1:], alpha=0.6)
    axes[0, 1].set_title('Error(t) vs Error(t-1)')
    axes[0, 1].set_xlabel('Error(t-1)')
    axes[0, 1].set_ylabel('Error(t)')
    
    # Añadir línea de regresión
    z = np.polyfit(errores[:-1], errores[1:], 1)
    p = np.poly1d(z)
    axes[0, 1].plot(sorted(errores[:-1]), p(sorted(errores[:-1])), "r--", alpha=0.8)
    
    # Añadir valor estimado de ρ y valor verdadero
    rho_est = np.corrcoef(errores[:-1], errores[1:])[0, 1]
    axes[0, 1].annotate(f'ρ est. = {rho_est:.3f}\nρ real = {rho_true:.3f}', 
                       xy=(0.05, 0.95), xycoords='axes fraction',
                       bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.8))
    
    # Autocorrelación
    plot_acf(errores, ax=axes[1, 0], lags=40)
    axes[1, 0].set_title('Función de Autocorrelación (ACF)')
    
    # Autocorrelación parcial
    plot_pacf(errores, ax=axes[1, 1], lags=40)
    axes[1, 1].set_title('Función de Autocorrelación Parcial (PACF)')
    
    plt.tight_layout()
    return fig

def visualizar_series_temporales(df):
    """
    Visualiza datos de series temporales con tendencia y estacionalidad.
    
    Parámetros:
    -----------
    df : DataFrame
        DataFrame con las variables generadas
        
    Retorna:
    --------
    fig : objeto Figure de matplotlib
        Figura con las visualizaciones
    """
    fig, axes = plt.subplots(3, 1, figsize=(14, 12))
    
    # Serie temporal completa
    axes[0].plot(df['tiempo'], df['Y'], label='Y')
    axes[0].set_title('Serie temporal completa')
    axes[0].set_xlabel('Tiempo')
    axes[0].set_ylabel('Y')
    axes[0].legend()
    
    # Componentes
    axes[1].plot(df['tiempo'], df['Y_sin_error'], label='Y sin error')
    axes[1].plot(df['tiempo'], df['tendencia'], label='Tendencia')
    axes[1].plot(df['tiempo'], df['estacional'], label='Estacional')
    axes[1].set_title('Componentes de la serie temporal')
    axes[1].set_xlabel('Tiempo')
    axes[1].set_ylabel('Valor')
    axes[1].legend()
    
    # Componente de error
    axes[2].plot(df['tiempo'], df['error'])
    axes[2].set_title('Componente de error (autocorrelacionado)')
    axes[2].set_xlabel('Tiempo')
    axes[2].set_ylabel('Error')
    
    plt.tight_layout()
    return fig

# Generar datos simples con autocorrelación
rho_true = 0.7
X, y, errores = generar_datos_ar1(n=200, rho=rho_true, seed=42)

# Mostrar estadísticas descriptivas
print("Estadísticas descriptivas de las variables generadas:")
descripcion = pd.concat([X, y, errores], axis=1).describe()
print(descripcion)

# Visualizar datos generados
fig1 = visualizar_datos_autocorrelacion(X, y, errores, rho_true)
plt.show()

# Generar datos de series temporales con tendencia y estacionalidad
df_series = generar_datos_tendencia_estacionalidad(n=200, rho=rho_true)

# Visualizar series temporales
fig2 = visualizar_series_temporales(df_series)
plt.show()


# Implementación de Modelos con Datos Sintéticos

A continuación, implementaremos y compararemos diferentes modelos para corregir la autocorrelación usando los datos sintéticos generados.


In [ ]:
from scipy.linalg import toeplitz

# ========== ETAPA 1: Modelo OLS ==========
print("="*50)
print("ANÁLISIS DE DATOS CON AUTOCORRELACIÓN AR(1)")
print("="*50)

print("\n1. Modelo OLS estándar (ignora autocorrelación)")
X_sm = sm.add_constant(X)  # aseguramos diseño
modelo_ols = sm.OLS(y, X_sm).fit()
print(modelo_ols.summary())

# Diagnóstico
resultados_diag = diagnostico_autocorrelacion(modelo_ols)
reportar_diagnostico_autocorrelacion(resultados_diag)

# Visualización
fig_acorr = visualizar_autocorrelacion(modelo_ols.resid)
plt.show()

# ========== ETAPA 2: Cochrane-Orcutt ==========
print("\n" + "="*50)
print("2. MÉTODO DE COCHRANE-ORCUTT")
print("="*50)
resultados_co = cochrane_orcutt(y, X)
reportar_resultados_correccion(resultados_co, metodo="Cochrane-Orcutt")

# ========== ETAPA 3: Prais-Winsten ==========
print("\n" + "="*50)
print("3. MÉTODO DE PRAIS-WINSTEN")
print("="*50)
resultados_pw = prais_winsten(y, X)
reportar_resultados_correccion(resultados_pw, metodo="Prais-Winsten")

# ========== ETAPA 4: Comparación ==========
print("\n" + "="*50)
print("4. COMPARACIÓN DE MÉTODOS")
print("="*50)
comparacion = comparar_modelos_autocorrelacion(y, X)
print(comparacion)

# Visualización de convergencia de ρ
plt.figure(figsize=(10, 6))
plt.plot(resultados_co['rhos_historia'], marker='o', label='Cochrane-Orcutt')
plt.plot(resultados_pw['rhos_historia'], marker='s', label='Prais-Winsten')
plt.axhline(y=rho_true, color='r', linestyle='--', label=f'Valor real (ρ={rho_true})')
plt.xlabel('Iteración')
plt.ylabel('Estimación de ρ')
plt.title('Convergencia de ρ en métodos iterativos')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# ========== ETAPA 5: GLS con matriz AR(1) simulada ==========
print("\n" + "="*50)
print("5. MODELO GLS CON ESTRUCTURA AR(1)")
print("="*50)

# Construcción de matriz de covarianza AR(1)
rho_gls = resultados_diag['acf'][1]
n = len(y)
sigma_ar1 = toeplitz(rho_gls ** np.arange(n))

# Ajustar modelo GLS
modelo_gls = sm.GLS(y, X_sm, sigma=sigma_ar1).fit()
print(modelo_gls.summary())

# Diagnóstico sobre residuos GLS
resultados_diag_gls = diagnostico_autocorrelacion(modelo_gls)
reportar_diagnostico_autocorrelacion(resultados_diag_gls)

# Visualización comparativa
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(modelo_ols.resid)
axes[0, 0].set_title('Residuos OLS')
axes[0, 0].set_xlabel('Tiempo')
axes[0, 0].set_ylabel('Residuo')

plot_acf(modelo_ols.resid, ax=axes[0, 1], lags=40)
axes[0, 1].set_title('ACF de Residuos OLS')

axes[1, 0].plot(modelo_gls.resid)
axes[1, 0].set_title('Residuos GLS')
axes[1, 0].set_xlabel('Tiempo')
axes[1, 0].set_ylabel('Residuo')

plot_acf(modelo_gls.resid, ax=axes[1, 1], lags=40)
axes[1, 1].set_title('ACF de Residuos GLS')

plt.tight_layout()
plt.show()


# Análisis de Series Temporales con Tendencia y Estacionalidad

Ahora exploraremos cómo manejar la autocorrelación en datos de series temporales que presentan tendencia y estacionalidad.


In [ ]:

# Usar los datos de series temporales generados
print("="*50)
print("ANÁLISIS DE SERIES TEMPORALES CON TENDENCIA Y ESTACIONALIDAD")
print("="*50)

# Preparar datos
X_series = df_series[['X1']]
y_series = df_series['Y']

# 1. Modelo OLS estándar (ignorando estructura temporal)
print("\n1. Modelo OLS estándar (ignora estructura temporal)")
modelo_ols_series = sm.OLS(y_series, sm.add_constant(X_series)).fit()
print(modelo_ols_series.summary())

# Diagnóstico de autocorrelación
resultados_diag_series = diagnostico_autocorrelacion(modelo_ols_series)
reportar_diagnostico_autocorrelacion(resultados_diag_series)

# Visualizar autocorrelación
fig_acorr_series = visualizar_autocorrelacion(modelo_ols_series.resid)
plt.show()

# 2. Modelo OLS con variables de tendencia y estacionalidad
print("\n" + "="*50)
print("2. MODELO OLS CON TENDENCIA Y ESTACIONALIDAD")
print("="*50)

# Crear variables para tendencia y estacionalidad
X_series_ext = X_series.copy()
# Añadir tendencia
X_series_ext['tendencia'] = df_series['tiempo']
# Añadir variables dummy estacionales para un periodo de 12 (por ejemplo, meses)
periodo = 12
for i in range(1, periodo):
    X_series_ext[f'estacional_{i}'] = (df_series['tiempo'] % periodo == i).astype(int)

# Ajustar modelo con tendencia y estacionalidad
modelo_ext = sm.OLS(y_series, sm.add_constant(X_series_ext)).fit()
print(modelo_ext.summary())

# Diagnóstico de autocorrelación
resultados_diag_ext = diagnostico_autocorrelacion(modelo_ext)
reportar_diagnostico_autocorrelacion(resultados_diag_ext)

# 3. Modelo con corrección AR(1) utilizando Prais-Winsten
print("\n" + "="*50)
print("3. PRAIS-WINSTEN CON TENDENCIA Y ESTACIONALIDAD")
print("="*50)
resultados_pw_series = prais_winsten(y_series, X_series_ext)
reportar_resultados_correccion(resultados_pw_series, metodo="Prais-Winsten")

# 4. Modelo ARIMA con regresores exógenos (ARIMAX)
print("\n" + "="*50)
print("4. MODELO ARIMA CON REGRESORES EXÓGENOS (ARIMAX)")
print("="*50)

# Preparar datos para SARIMAX
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Ajustar modelo ARIMA(1,0,0) con regresores (equivalente a AR(1) con X)
modelo_arimax = SARIMAX(y_series, 
                        exog=sm.add_constant(X_series),
                        order=(1, 0, 0),  # (p, d, q)
                        seasonal_order=(0, 0, 0, 0),  # No componente estacional en el ARIMA
                        enforce_stationarity=False).fit(disp=False)

print(modelo_arimax.summary())

# Comparar coeficientes entre modelos
print("\n" + "="*50)
print("COMPARACIÓN DE COEFICIENTES ENTRE MODELOS")
print("="*50)

print(f"{'Parámetro':<15} {'OLS':<12} {'OLS+Tend+Est':<12} {'Prais-Winsten':<12} {'ARIMAX':<12}")
print("-" * 60)

# Coeficientes de OLS
coef_ols = modelo_ols_series.params
# Coeficientes de OLS con tendencia y estacionalidad (solo los primeros)
coef_ext = modelo_ext.params
# Coeficientes de Prais-Winsten
coef_pw = resultados_pw_series['modelo_final'].params
# Coeficientes de ARIMAX
coef_arimax = modelo_arimax.params

# Comparar intercepto
print(f"{'Intercepto':<15} {coef_ols[0]:<12.4f} {coef_ext[0]:<12.4f} {coef_pw[0]:<12.4f} {coef_arimax[0]:<12.4f}")
# Comparar beta de X1
print(f"{'X1':<15} {coef_ols[1]:<12.4f} {coef_ext[1]:<12.4f} {coef_pw[1]:<12.4f} {coef_arimax[1]:<12.4f}")
# Coeficiente AR(1) para ARIMAX
print(f"{'AR(1)':<15} {'N/A':<12} {'N/A':<12} {resultados_pw_series['rho_final']:<12.4f} {coef_arimax[2]:<12.4f}")

# Comparar residuos
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Residuos OLS
axes[0, 0].plot(modelo_ols_series.resid)
axes[0, 0].set_title('Residuos OLS')
axes[0, 0].set_xlabel('Tiempo')
axes[0, 0].set_ylabel('Residuo')

# Residuos OLS con tendencia y estacionalidad
axes[0, 1].plot(modelo_ext.resid)
axes[0, 1].set_title('Residuos OLS con Tendencia y Estacionalidad')
axes[0, 1].set_xlabel('Tiempo')
axes[0, 1].set_ylabel('Residuo')

# Residuos ARIMAX
axes[1, 0].plot(modelo_arimax.resid)
axes[1, 0].set_title('Residuos ARIMAX')
axes[1, 0].set_xlabel('Tiempo')
axes[1, 0].set_ylabel('Residuo')

# ACF de residuos ARIMAX
plot_acf(modelo_arimax.resid, ax=axes[1, 1], lags=40)
axes[1, 1].set_title('ACF de Residuos ARIMAX')

plt.tight_layout()
plt.show()

# Predicciones y ajuste del modelo
fig, ax = plt.subplots(figsize=(12, 6))

# Datos originales
ax.plot(df_series['tiempo'], y_series, 'k-', label='Datos Originales')

# Predicciones de diferentes modelos
ax.plot(df_series['tiempo'], modelo_ols_series.predict(sm.add_constant(X_series)), 'b--', label='OLS')
ax.plot(df_series['tiempo'], modelo_ext.predict(sm.add_constant(X_series_ext)), 'g--', label='OLS + Tendencia + Estacionalidad')
ax.plot(df_series['tiempo'], modelo_arimax.predict(), 'r--', label='ARIMAX')

ax.set_title('Comparación de Predicciones entre Modelos')
ax.set_xlabel('Tiempo')
ax.set_ylabel('Y')
ax.legend()
plt.grid(True, alpha=0.3)
plt.show()


# Comparativa de Errores de Predicción

Una parte importante del análisis de autocorrelación es evaluar cómo afecta a la capacidad predictiva del modelo y cómo las correcciones mejoran las predicciones.


In [ ]:
# ========================================
# Evaluación de capacidad predictiva
# ========================================
print("="*50)
print("EVALUACIÓN DE CAPACIDAD PREDICTIVA")
print("="*50)

def calcular_metricas_prediccion(y_true, y_pred):
    """
    Calcula métricas de error de predicción.
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    errores = y_true - y_pred
    mse = np.mean(errores ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(errores))
    mape = np.mean(np.abs(errores / y_true)) * 100
    
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

# ========================================
# 1. Predicción dentro de muestra (datos AR(1))
# ========================================
print("\n1. Predicción dentro de muestra para datos AR(1)")

X_sm = sm.add_constant(X)

# OLS
y_pred_ols = modelo_ols.predict(X_sm)
metricas_ols = calcular_metricas_prediccion(y, y_pred_ols)

# Cochrane-Orcutt
rho_co = resultados_co['rho_final']
X_co_raw = X_sm[1:] - rho_co * X_sm[:-1]
X_co = pd.DataFrame(
    X_co_raw,
    columns=X_sm.columns,
    index=np.arange(len(y) - 1)  # índice numérico compatible
)
preds_co = resultados_co['modelo_final'].predict(X_co)
y_pred_co = np.full_like(y, fill_value=np.nan)
y_pred_co[1:] = preds_co
metricas_co = calcular_metricas_prediccion(y[1:], y_pred_co[1:])

# Prais-Winsten
rho_pw = resultados_pw['rho_final']
X_pw_0 = X_sm.iloc[0:1] * np.sqrt(1 - rho_pw**2)
X_pw_rest = X_sm.iloc[1:] - rho_pw * X_sm.iloc[:-1].values
X_pw = pd.concat([X_pw_0, pd.DataFrame(X_pw_rest, columns=X_sm.columns)], ignore_index=True)
y_pred_pw = resultados_pw['modelo_final'].predict(X_pw)
metricas_pw = calcular_metricas_prediccion(y, y_pred_pw)

# GLS
y_pred_gls = modelo_gls.predict(X_sm)
metricas_gls = calcular_metricas_prediccion(y, y_pred_gls)

# ========================================
# Consolidar resultados en un DataFrame
# ========================================
metricas_df = pd.DataFrame({
    'OLS': [metricas_ols['MSE'], metricas_ols['RMSE'], metricas_ols['MAE'], metricas_ols['MAPE']],
    'Cochrane-Orcutt': [metricas_co['MSE'], metricas_co['RMSE'], metricas_co['MAE'], metricas_co['MAPE']],
    'Prais-Winsten': [metricas_pw['MSE'], metricas_pw['RMSE'], metricas_pw['MAE'], metricas_pw['MAPE']],
    'GLS': [metricas_gls['MSE'], metricas_gls['RMSE'], metricas_gls['MAE'], metricas_gls['MAPE']]
}, index=['MSE', 'RMSE', 'MAE', 'MAPE'])

print(metricas_df)

# ========================================
# Visualización
# ========================================
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
metricas_df.loc[['RMSE', 'MAE']].T.plot(kind='bar', ax=ax)
ax.set_title('Métricas de Error - Modelos AR(1)')
ax.set_ylabel('Error')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


# Caso de Estudio con Datos Reales

Para consolidar los conceptos aprendidos, aplicaremos las técnicas de detección y corrección de autocorrelación a un conjunto de datos reales: series temporales de indicadores económicos.


In [ ]:

# Cargar datos reales
print("="*50)
print("CASO DE ESTUDIO: SERIES ECONÓMICAS")
print("="*50)

# Importar datos de paquetes de ejemplo o generarlos si no están disponibles
try:
    # Intento 1: Usar datos de macroeconomía de statsmodels
    import statsmodels.datasets as datasets
    data_macro = datasets.macrodata.load_pandas().data
    print("Datos macroeconómicos cargados correctamente.")
    
    # Crear DataFrame con los datos
    df_macro = data_macro[['year', 'quarter', 'realgdp', 'realcons', 'realinv', 'realgovt', 'cpi', 'unemp']].copy()
    
    # Crear una variable de tiempo continua
    df_macro['t'] = df_macro['year'] + df_macro['quarter']/4 - 1900
    
    # Ordenar por tiempo
    df_macro = df_macro.sort_values('t').reset_index(drop=True)
    
    # Calcular tasas de crecimiento
    df_macro['gdp_growth'] = df_macro['realgdp'].pct_change() * 100
    df_macro['cons_growth'] = df_macro['realcons'].pct_change() * 100
    df_macro['inv_growth'] = df_macro['realinv'].pct_change() * 100
    
    # Eliminar primera fila con NaN
    df_macro = df_macro.dropna().reset_index(drop=True)
    
except:
    try:
        # Intento 2: Usar datos de ejemplo de pandas
        import pandas as pd
        import pandas_datareader as pdr
        
        # Intentar cargar datos economicos
        print("Intentando cargar datos económicos de pandas-datareader...")
        start_date = '1990-01-01'
        end_date = '2022-12-31'
        
        # Intentar con diferentes fuentes
        try:
            df_fred = pdr.get_data_fred(['GDPC1', 'PCECC96', 'GPDIC1', 'CPIAUCSL', 'UNRATE'], 
                                     start=start_date, end=end_date)
            print("Datos económicos de FRED cargados correctamente.")
            
            # Renombrar columnas
            df_fred.columns = ['realgdp', 'realcons', 'realinv', 'cpi', 'unemp']
            
            # Resamplear a trimestral y calcular tasas de crecimiento
            df_macro = df_fred.resample('Q').mean()
            df_macro['year'] = df_macro.index.year
            df_macro['quarter'] = df_macro.index.quarter
            df_macro['t'] = df_macro.index.year + df_macro.index.quarter/4 - 1900
            
            # Calcular tasas de crecimiento
            df_macro['gdp_growth'] = df_macro['realgdp'].pct_change() * 100
            df_macro['cons_growth'] = df_macro['realcons'].pct_change() * 100
            df_macro['inv_growth'] = df_macro['realinv'].pct_change() * 100
            
            # Eliminar filas con NaN
            df_macro = df_macro.dropna().reset_index(drop=True)
            
        except:
            raise Exception("No se pudieron cargar datos económicos.")
    
    except:
        # Intento 3: Generar datos sintéticos con estructura similar
        print("Generando datos económicos sintéticos...")
        
        np.random.seed(123)
        n = 200  # Número de trimestres (50 años)
        
        # Crear fechas trimestrales
        años = np.arange(1960, 1960 + n//4 + 1)
        trimestres = np.tile([1, 2, 3, 4], len(años))[:n]
        
        # Variable de tiempo
        t = np.arange(n)
        
        # Generar PIB con tendencia y ciclo
        tendencia_pib = 5000 + 50 * t + 0.1 * t**2
        ciclo_pib = 500 * np.sin(2 * np.pi * t / 20)  # Ciclo de 5 años
        error_pib = np.zeros(n)
        
        # Proceso AR(1) para el error
        rho_pib = 0.8
        for i in range(1, n):
            error_pib[i] = rho_pib * error_pib[i-1] + np.random.normal(0, 100)
        
        realgdp = tendencia_pib + ciclo_pib + error_pib
        
        # Generar consumo (aproximadamente 60-70% del PIB con ruido)
        realcons = 0.65 * realgdp + np.random.normal(0, 100, n)
        
        # Generar inversión (aproximadamente 15-25% del PIB con más volatilidad)
        realinv = 0.2 * realgdp + 200 * np.sin(2 * np.pi * t / 16) + np.random.normal(0, 150, n)
        
        # Generar gasto gubernamental
        realgovt = 0.15 * realgdp + 100 * np.sin(2 * np.pi * t / 24) + np.random.normal(0, 80, n)
        
        # Generar inflación (CPI)
        cpi_base = 30 + 0.2 * t  # Tendencia base
        cpi_cycle = 5 * np.sin(2 * np.pi * t / 32)  # Ciclo largo
        error_cpi = np.zeros(n)
        
        # Proceso AR(1) para el error
        rho_cpi = 0.7
        for i in range(1, n):
            error_cpi[i] = rho_cpi * error_cpi[i-1] + np.random.normal(0, 1)
        
        cpi = cpi_base + cpi_cycle + error_cpi
        
        # Generar desempleo (correlación negativa con PIB)
        unemp_base = 6 + np.random.normal(0, 0.5, n)  # Base alrededor del 6%
        unemp_cycle = -0.003 * ciclo_pib  # Relación inversa con ciclo económico
        error_unemp = np.zeros(n)
        
        # Proceso AR(1) para el error
        rho_unemp = 0.9  # Alta persistencia en desempleo
        for i in range(1, n):
            error_unemp[i] = rho_unemp * error_unemp[i-1] + np.random.normal(0, 0.2)
        
        unemp = np.maximum(2, unemp_base + unemp_cycle + error_unemp)  # Mínimo 2%
        
        # Calcular tasas de crecimiento
        gdp_growth = np.zeros(n)
        cons_growth = np.zeros(n)
        inv_growth = np.zeros(n)
        
        for i in range(1, n):
            gdp_growth[i] = (realgdp[i] / realgdp[i-1] - 1) * 100
            cons_growth[i] = (realcons[i] / realcons[i-1] - 1) * 100
            inv_growth[i] = (realinv[i] / realinv[i-1] - 1) * 100
        
        # Crear DataFrame
        df_macro = pd.DataFrame({
            'year': 1960 + t // 4,
            'quarter': trimestres,
            't': t / 4 + 1960,
            'realgdp': realgdp,
            'realcons': realcons,
            'realinv': realinv,
            'realgovt': realgovt,
            'cpi': cpi,
            'unemp': unemp,
            'gdp_growth': gdp_growth,
            'cons_growth': cons_growth,
            'inv_growth': inv_growth
        })
        
        # Eliminar primera fila con NaN en tasas de crecimiento
        df_macro = df_macro.iloc[1:].reset_index(drop=True)

# Mostrar información sobre los datos
print(f"Dimensiones del DataFrame: {df_macro.shape}")
print("\nPrimeras filas:")
print(df_macro.head())

print("\nEstadísticas descriptivas:")
print(df_macro.describe())

# Visualizar series temporales
fig, axes = plt.subplots(3, 2, figsize=(16, 12))

# PIB real
axes[0, 0].plot(df_macro['t'], df_macro['realgdp'])
axes[0, 0].set_title('PIB Real')
axes[0, 0].set_xlabel('Año')
axes[0, 0].set_ylabel('PIB')
axes[0, 0].grid(True, alpha=0.3)

# Crecimiento del PIB
axes[0, 1].plot(df_macro['t'], df_macro['gdp_growth'])
axes[0, 1].set_title('Crecimiento del PIB (%)')
axes[0, 1].set_xlabel('Año')
axes[0, 1].set_ylabel('Crecimiento (%)')
axes[0, 1].grid(True, alpha=0.3)

# Consumo real
axes[1, 0].plot(df_macro['t'], df_macro['realcons'])
axes[1, 0].set_title('Consumo Real')
axes[1, 0].set_xlabel('Año')
axes[1, 0].set_ylabel('Consumo')
axes[1, 0].grid(True, alpha=0.3)

# Inversión real
axes[1, 1].plot(df_macro['t'], df_macro['realinv'])
axes[1, 1].set_title('Inversión Real')
axes[1, 1].set_xlabel('Año')
axes[1, 1].set_ylabel('Inversión')
axes[1, 1].grid(True, alpha=0.3)

# Inflación (CPI)
axes[2, 0].plot(df_macro['t'], df_macro['cpi'])
axes[2, 0].set_title('Índice de Precios al Consumidor')
axes[2, 0].set_xlabel('Año')
axes[2, 0].set_ylabel('CPI')
axes[2, 0].grid(True, alpha=0.3)

# Desempleo
axes[2, 1].plot(df_macro['t'], df_macro['unemp'])
axes[2, 1].set_title('Tasa de Desempleo (%)')
axes[2, 1].set_xlabel('Año')
axes[2, 1].set_ylabel('Desempleo (%)')
axes[2, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Modelo 1: Relación entre crecimiento del PIB y desempleo (Ley de Okun)
print("\n" + "="*50)
print("MODELO 1: LEY DE OKUN (RELACIÓN PIB-DESEMPLEO)")
print("="*50)

# Variables para el modelo
X_okun = sm.add_constant(df_macro['gdp_growth'])
y_okun = df_macro['unemp'].diff()  # Cambio en la tasa de desempleo

# Eliminar valores NaN
mask_okun = ~np.isnan(y_okun)
X_okun = X_okun[mask_okun]
y_okun = y_okun[mask_okun]

# Ajustar modelo OLS
modelo_okun_ols = sm.OLS(y_okun, X_okun).fit()
print(modelo_okun_ols.summary())

# Diagnóstico de autocorrelación
resultados_diag_okun = diagnostico_autocorrelacion(modelo_okun_ols)
reportar_diagnostico_autocorrelacion(resultados_diag_okun)

# Visualizar autocorrelación en residuos
fig_acorr_okun = visualizar_autocorrelacion(modelo_okun_ols.resid)
plt.show()

# Aplicar método de Prais-Winsten
print("\nAplicando método de Prais-Winsten a la Ley de Okun:")
resultados_pw_okun = prais_winsten(y_okun, X_okun.iloc[:, 1:])
reportar_resultados_correccion(resultados_pw_okun, metodo="Prais-Winsten")

# Modelo 2: Relación entre inflación y desempleo (Curva de Phillips)
print("\n" + "="*50)
print("MODELO 2: CURVA DE PHILLIPS (RELACIÓN INFLACIÓN-DESEMPLEO)")
print("="*50)

# Calcular inflación como cambio porcentual en CPI
df_macro['inflation'] = df_macro['cpi'].pct_change() * 100

# Variables para el modelo
X_phillips = sm.add_constant(df_macro['unemp'])
y_phillips = df_macro['inflation']

# Eliminar valores NaN
mask_phillips = ~np.isnan(y_phillips)
X_phillips = X_phillips[mask_phillips]
y_phillips = y_phillips[mask_phillips]

# Ajustar modelo OLS
modelo_phillips_ols = sm.OLS(y_phillips, X_phillips).fit()
print(modelo_phillips_ols.summary())

# Diagnóstico de autocorrelación
resultados_diag_phillips = diagnostico_autocorrelacion(modelo_phillips_ols)
reportar_diagnostico_autocorrelacion(resultados_diag_phillips)

# Visualizar autocorrelación en residuos
fig_acorr_phillips = visualizar_autocorrelacion(modelo_phillips_ols.resid)
plt.show()

# Aplicar método de Prais-Winsten
print("\nAplicando método de Prais-Winsten a la Curva de Phillips:")
resultados_pw_phillips = prais_winsten(y_phillips, X_phillips.iloc[:, 1:])
reportar_resultados_correccion(resultados_pw_phillips, metodo="Prais-Winsten")

# Modelo 3: Modelo de inversión 
print("\n" + "="*50)
print("MODELO 3: MODELO DE INVERSIÓN")
print("="*50)

# Variables para el modelo
X_inv = sm.add_constant(pd.DataFrame({
    'gdp_growth': df_macro['gdp_growth'],
    'lagged_inv_growth': df_macro['inv_growth'].shift(1),
    'unemp': df_macro['unemp']
}))
y_inv = df_macro['inv_growth']

# Eliminar valores NaN
mask_inv = ~(np.isnan(y_inv) | X_inv.isna().any(axis=1))
X_inv = X_inv[mask_inv]
y_inv = y_inv[mask_inv]

# Ajustar modelo OLS
modelo_inv_ols = sm.OLS(y_inv, X_inv).fit()
print(modelo_inv_ols.summary())

# Diagnóstico de autocorrelación
resultados_diag_inv = diagnostico_autocorrelacion(modelo_inv_ols)
reportar_diagnostico_autocorrelacion(resultados_diag_inv)

# Visualizar autocorrelación en residuos
fig_acorr_inv = visualizar_autocorrelacion(modelo_inv_ols.resid)
plt.show()

# Aplicar método de Prais-Winsten
print("\nAplicando método de Prais-Winsten al Modelo de Inversión:")
resultados_pw_inv = prais_winsten(y_inv, X_inv.iloc[:, 1:])
reportar_resultados_correccion(resultados_pw_inv, metodo="Prais-Winsten")

# Comparación de los tres modelos antes y después de la corrección
print("\n" + "="*50)
print("COMPARACIÓN DE MODELOS ANTES Y DESPUÉS DE CORRECCIÓN")
print("="*50)

modelos = ['Ley de Okun', 'Curva de Phillips', 'Modelo de Inversión']
dw_ols = [durbin_watson(modelo_okun_ols.resid), durbin_watson(modelo_phillips_ols.resid), durbin_watson(modelo_inv_ols.resid)]
dw_pw = [durbin_watson(resultados_pw_okun['modelo_final'].resid), durbin_watson(resultados_pw_phillips['modelo_final'].resid), durbin_watson(resultados_pw_inv['modelo_final'].resid)]
rho_pw = [resultados_pw_okun['rho_final'], resultados_pw_phillips['rho_final'], resultados_pw_inv['rho_final']]

df_comparacion = pd.DataFrame({
    'Modelo': modelos,
    'DW OLS': dw_ols,
    'DW Prais-Winsten': dw_pw,
    'Rho Estimado': rho_pw
})

print(df_comparacion)

# Visualizar resultados de la corrección
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Durbin-Watson antes y después
width = 0.35
x = np.arange(len(modelos))
axes[0].bar(x - width/2, dw_ols, width, label='OLS')
axes[0].bar(x + width/2, dw_pw, width, label='Prais-Winsten')
axes[0].axhline(y=2.0, color='r', linestyle='--', alpha=0.7)
axes[0].set_xticks(x)
axes[0].set_xticklabels(modelos)
axes[0].set_ylabel('Estadístico Durbin-Watson')
axes[0].set_title('DW antes y después de corrección')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Rho estimado
axes[1].bar(modelos, rho_pw)
axes[1].set_ylabel('Rho Estimado')
axes[1].set_title('Coeficientes de Autocorrelación Estimados')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# Ejercicios Prácticos

A continuación se proponen varios ejercicios para reforzar los conceptos aprendidos sobre autocorrelación y series temporales.

### Ejercicio 1: Detección de Autocorrelación

Utiliza el siguiente código para generar un conjunto de datos con autocorrelación:

```python
np.random.seed(123)
n = 100
X = np.random.normal(0, 1, n)
# Generar errores con autocorrelación AR(1)
errores = np.zeros(n)
rho = 0.8
for t in range(1, n):
    errores[t] = rho * errores[t-1] + np.random.normal(0, 1)
y = 2 + 3 * X + errores
```

1. Calcula manualmente el estadístico de Durbin-Watson y compáralo con la función de statsmodels.
2. Implementa un test de hipótesis para la autocorrelación de primer orden usando el estadístico de Durbin-Watson.
3. Estima el coeficiente de autocorrelación (ρ) mediante regresión de los residuos contra sus rezagos.
4. Visualiza la función de autocorrelación (ACF) y la función de autocorrelación parcial (PACF) e interpreta los resultados.

### Ejercicio 2: Corrección de Autocorrelación

Utilizando los datos del Ejercicio 1:

1. Implementa manualmente el método de Cochrane-Orcutt con 5 iteraciones, mostrando los coeficientes y ρ en cada iteración.
2. Implementa manualmente el método de Prais-Winsten para los mismos datos.
3. Compara los resultados de ambos métodos con los obtenidos mediante GLS de statsmodels.
4. Analiza los residuos después de la corrección y verifica si se ha eliminado la autocorrelación.

### Ejercicio 3: Simulación de Efectos de Autocorrelación

1. Escribe una función que genere datos con diferentes valores de ρ (desde -0.9 hasta 0.9 en incrementos de 0.3).
2. Para cada conjunto de datos, ajusta un modelo OLS y calcula:
   - La varianza de los coeficientes estimados
   - El error de predicción cuadrático medio
   - El estadístico de Durbin-Watson
3. Visualiza cómo varían estas métricas en función de ρ y extrae conclusiones sobre los efectos de la autocorrelación.

### Ejercicio 4: Series Temporales con Tendencia y Estacionalidad

Crea datos con tendencia y estacionalidad utilizando el siguiente código:

```python
np.random.seed(456)
n = 120  # 10 años de datos mensuales
t = np.arange(n)
# Tendencia
tendencia = 0.2 * t
# Estacionalidad (ciclo anual)
estacionalidad = 2 * np.sin(2 * np.pi * t / 12)
# Componente AR(1)
ar = np.zeros(n)
rho = 0.7
for i in range(1, n):
    ar[i] = rho * ar[i-1] + np.random.normal(0, 1)
# Combinar componentes
y = 10 + tendencia + estacionalidad + ar
```

1. Divide la serie en sus componentes (tendencia, estacionalidad, error) utilizando regresión.
2. Ajusta un modelo con variables dummy estacionales y tendencia lineal.
3. Analiza los residuos para detectar autocorrelación.
4. Aplica GLS con estructura AR(1) y compara con los resultados OLS.
5. Genera predicciones con ambos modelos y evalúa su precisión.


# Conclusiones

En este taller hemos explorado el fenómeno de la autocorrelación en modelos lineales y series temporales. Los principales conceptos y aprendizajes incluyen:

1. **Naturaleza de la Autocorrelación**:
   - La autocorrelación ocurre cuando los errores de un modelo están correlacionados a lo largo del tiempo o secuencia.
   - Es común en datos de series temporales y datos espaciales, pero puede aparecer en cualquier conjunto de datos con estructura secuencial.
   - El modelo AR(1) es la forma más común de modelar la autocorrelación: $\varepsilon_t = \rho \varepsilon_{t-1} + u_t$.

2. **Consecuencias de Ignorar la Autocorrelación**:
   - Los estimadores OLS siguen siendo insesgados y consistentes, pero ya no son eficientes.
   - Los errores estándar están mal estimados, lo que conduce a inferencias incorrectas.
   - Las pruebas de hipótesis (t, F) y los intervalos de confianza son inválidos.
   - La predicción es subóptima, especialmente para observaciones futuras.

3. **Métodos de Detección**:
   - El estadístico de Durbin-Watson es fundamental para detectar autocorrelación de primer orden.
   - Las funciones ACF y PACF proporcionan información sobre la estructura y orden de la autocorrelación.
   - Pruebas como Breusch-Godfrey y Ljung-Box permiten detectar autocorrelación de orden superior.

4. **Métodos de Corrección**:
   - El método de Cochrane-Orcutt transforma las variables para eliminar la autocorrelación, pero pierde la primera observación.
   - El método de Prais-Winsten mejora sobre Cochrane-Orcutt al preservar la primera observación.
   - Los Mínimos Cuadrados Generalizados (GLS) proporcionan un marco más general para manejar diferentes estructuras de correlación.
   - Los modelos ARIMA integran la corrección de autocorrelación dentro de la estructura del modelo.

5. **Series Temporales**:
   - Las series temporales suelen presentar componentes de tendencia, estacionalidad y errores autocorrelacionados.
   - Los modelos pueden incluir variables dummy estacionales, tendencias determinísticas y términos autorregresivos.
   - La estacionariedad es un concepto clave para el análisis adecuado de series temporales.

La correcta identificación y manejo de la autocorrelación es esencial para obtener inferencias válidas y predicciones precisas en modelos con estructura temporal. Los métodos aprendidos en este taller proporcionan un conjunto de herramientas para abordar este desafío común en el análisis de datos económicos, financieros, ambientales y de muchos otros campos.


# Referencias

1. Box, G. E. P., Jenkins, G. M., Reinsel, G. C., & Ljung, G. M. (2015). *Time Series Analysis: Forecasting and Control* (5th ed.). Wiley.

2. Cochrane, D., & Orcutt, G. H. (1949). Application of Least Squares Regression to Relationships Containing Auto-Correlated Error Terms. *Journal of the American Statistical Association, 44*(245), 32-61.

3. Durbin, J., & Watson, G. S. (1950). Testing for Serial Correlation in Least Squares Regression: I. *Biometrika, 37*(3/4), 409-428.

4. Greene, W. H. (2018). *Econometric Analysis* (8th ed.). Pearson.

5. Hamilton, J. D. (2020). *Time Series Analysis*. Princeton University Press.

6. Ljung, G. M., & Box, G. E. P. (1978). On a Measure of Lack of Fit in Time Series Models. *Biometrika, 65*(2), 297-303.

7. Prais, S. J., & Winsten, C. B. (1954). Trend Estimators and Serial Correlation. *Cowles Commission Discussion Paper, No. 383*, Chicago.

8. Wooldridge, J. M. (2020). *Introductory Econometrics: A Modern Approach* (7th ed.). Cengage Learning.
